# Reward-Aware Population Scaling of ES in LLM Fine-Tuning — Demo

**STAT 4830 — Spring 2026**, University of Pennsylvania  
Sung Cho · Gyubin Han · Maxwell DeLorenzo

This notebook is an **executable companion** to the project. It is *not* a heavyweight reproduction — full ES fine-tuning takes hours on a real GPU. Instead, this notebook is the most efficient way to:

1. Get the intuition for the three claims of the project in code, on the CPU, in under a minute.
2. See the actual figures from the report and slides without flipping PDFs.
3. Optionally run the **zero-training degeneracy probe** on a real LLM (T4-class GPU, a few minutes).
4. Find the right CLI command for whatever you want to reproduce next.

Source of truth: [`final-report.pdf`](final-report.pdf) (theoretical, ICM workshop) and [`final-slides.pdf`](final-slides.pdf) (broader experimental story). Full setup and a longer command list is in [`../README.md`](../README.md).

**Assumed environment:** the repo's `uv sync` has been run and the venv is active (kernel = `.venv/bin/python`). No install cells.

---

## 1. The motivating paradox

Two recent papers fine-tune LLMs with **literally the same ES estimator** and reach **opposite** conclusions about the population size `N`:

| Paper | Reward | Recommended `N` |
|---|---|---|
| **MeZO** (Malladi et al., 2023) | Cross-entropy (white-box logits) | `N = 1` |
| **ES-at-Scale** (Qiu et al., 2026) | Binary accuracy (black-box) | `N ≈ 30` |

We argue this is **not a paradox** but a *scaling law*: the right `N` is set by the **reward function**, not by the model or the task. The rest of this notebook is a guided tour through the three reasons why.

---

## 2. The ES estimator in 5 lines

For parameters `θ ∈ ℝ^d`, antithetic ES estimates the gradient of the Gaussian-smoothed objective `f_σ(θ) = 𝔼_ε[f(θ + σε)]` via:

$$\hat g \;=\; \frac{1}{N\sigma} \sum_{i=1}^{N} \bigl[R(\theta + \sigma\varepsilon_i, \mathcal{B}) - R(\theta - \sigma\varepsilon_i, \mathcal{B})\bigr]\,\varepsilon_i,\qquad \varepsilon_i \sim \mathcal{N}(0, I_d)$$

$$\theta \;\leftarrow\; \theta + \eta\, \hat g$$

It is **memory-efficient** (no activations), **embarrassingly parallel** (independent seeds), and **black-box compatible** (`R` need not be differentiable).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

def es_step(theta, R, sigma, eta, N, normalize=False, rng=rng):
    """One antithetic ES step. Returns updated theta and the population's pair-level advantages."""
    eps = rng.normal(size=(N, *theta.shape))
    A = np.array([R(theta + sigma * e) - R(theta - sigma * e) for e in eps])
    if normalize:
        A_hat = (A - A.mean()) / (A.std() + 1e-12)
    else:
        A_hat = A
    g_hat = (1.0 / (N * sigma)) * (A_hat[:, None] * eps).sum(axis=0)
    return theta + eta * g_hat, A


## 3. Lesson 3 in four lines — z-score normalization erases scale at `N = 2`

Before we get to the trajectory demo, here is **Proposition 4** of the report stated as a one-liner: for two pair-level advantages `A_1, A_2`, the z-score normalization

$$\hat A_i \;=\; \frac{A_i - \bar A}{s_A + \varepsilon_0}$$

produces the vector `(±1, ∓1)` **regardless of |A_1 − A_2|**. The absolute advantage scale literally cancels.

In [ ]:
def zscore(a):
    a = np.asarray(a, dtype=float)
    return (a - a.mean()) / (a.std() + 1e-12)

for A1 in [1e-6, 1e-3, 1.0, 100.0]:
    A2 = 0.0
    print(f"|A1 - A2| = {abs(A1 - A2):>8.0e}   raw = {[A1, A2]}   z-score = {zscore([A1, A2])}")

print("\nIdentical normalized magnitude across 8 orders of advantage gap.")
print("=> raw ES is self-annealing near degeneracy; normalized ES is not.")


## 4. Toy demo: N = 2 ES with binary reward, normalize on vs off

We pose a 2-D linear-classification toy: 16 random examples in `ℝ²`, ground-truth label `sign(x₁)`, reward = fraction correctly classified by `sign(θ · x)`. The optimal `θ` is anywhere along the positive `x₁` axis. Starting at a misaligned `θ₀`, we run 80 steps of `N = 2` antithetic ES with `σ = 0.3`, `η = 0.1` under two settings:

- **Normalize OFF** (raw): updates shrink as advantages shrink — the self-annealing regime.
- **Normalize ON** (z-score): updates are fixed-magnitude — the small-N pathology.

Both runs share the **same** random seed for perturbations and batch indexing, so any divergence is entirely from the normalization step.

In [ ]:
data_rng = np.random.default_rng(0)
B = 16
X = data_rng.normal(size=(B, 2))
y = np.sign(X @ np.array([1.0, 0.0]))

def R(theta):
    pred = np.sign(X @ theta)
    return float(np.mean(pred == y))

def run(normalize):
    rng_local = np.random.default_rng(7)
    theta = np.array([-0.6, 0.8])
    traj = [theta.copy()]
    rewards = [R(theta)]
    for _ in range(80):
        theta, _ = es_step(theta, R, sigma=0.3, eta=0.1, N=2, normalize=normalize, rng=rng_local)
        traj.append(theta.copy())
        rewards.append(R(theta))
    return np.array(traj), np.array(rewards)

traj_off, rew_off = run(normalize=False)
traj_on,  rew_on  = run(normalize=True)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

ax = axes[0]
ax.plot(traj_off[:, 0], traj_off[:, 1], '-', color='C1', label='normalize OFF (raw)', lw=2)
ax.plot(traj_on[:, 0],  traj_on[:, 1],  '-', color='C0', label='normalize ON (z-score)', lw=2)
ax.scatter([traj_off[0, 0]], [traj_off[0, 1]], color='k', zorder=5, label='start')
ax.axvline(0, color='gray', lw=0.5)
ax.axhline(0, color='gray', lw=0.5)
ax.set_xlabel(r'$\theta_1$'); ax.set_ylabel(r'$\theta_2$')
ax.set_title('Parameter trajectory ($N=2$)')
ax.legend(loc='lower right')
ax.set_aspect('equal'); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(rew_off, color='C1', label='normalize OFF (raw)', lw=2)
ax.plot(rew_on,  color='C0', label='normalize ON (z-score)', lw=2)
ax.set_xlabel('step'); ax.set_ylabel('reward (val acc)')
ax.set_title('Reward curve ($N=2$, same seed, same data)')
ax.legend(loc='lower right'); ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final reward (normalize OFF): {rew_off[-1]:.3f}")
print(f"Final reward (normalize ON):  {rew_on[-1]:.3f}")


## 5. Lesson 1: reward granularity sets the availability threshold `N_avail`

A seed pair is **degenerate** when both perturbations give exactly the same batch reward — it contributes zero signal to the ES gradient. Let `q = P(A = 0)` and let `K_N = #{i : A_i ≠ 0}` count surviving seeds. Then

$$\Pr(K_N = 0) \;=\; q^N, \qquad N_{\text{avail}}(\delta) \;=\; \bigl\lceil \log\delta\,/\,\log q \bigr\rceil.$$

- **CE reward** (continuous): `q = 0` almost surely → `N_avail = 1`. MeZO works at `N=1` *because* CE has no degeneracy. This is **Corollary 2** in the report.
- **Binary accuracy reward**: a positive `q` appears. The report's **Proposition 3** gives a closed-form approximation:

$$q \;\approx\; \frac{1}{\sqrt{4\pi B\, p_0\,(1 - p_0)\,(1 - \rho)}}$$

where `p_0` is the base-model accuracy, `B` is the batch size, and `ρ` is the intra-pair correctness correlation.

In [ ]:
import math

def q_formula(p0, B, rho=0.0):
    return 1.0 / math.sqrt(4 * math.pi * B * p0 * (1 - p0) * (1 - rho))

def n_avail(p0, B, rho=0.0, delta=0.05):
    q = min(0.999, q_formula(p0, B, rho))
    return math.ceil(math.log(delta) / math.log(q))

# Reproduces the table in Week-12 §4.1 / final-report §3 (with rho=0 'independent' special case)
print(f"{'p_0':>5} {'B':>3} {'rho':>5} {'q':>8} {'N_avail (δ=0.05)':>20}")
for p0, B, rho, label in [
    (0.50, 16, 0.00, 'Mid-training, binary task'),
    (0.17, 16, 0.00, 'Random init, 6-class task'),
    (0.10, 16, 0.00, 'Weak base model'),
    (0.01, 16, 0.00, 'Uninstructed base model'),
    (0.47, 16, 0.42, 'Qwen2.5-1.5B / GSM8K (our probe)'),
    (0.31, 16, 0.36, 'Qwen2.5-0.5B / GSM8K (our probe)'),
    (0.64,  4, 0.61, 'Qwen2.5-7B  / GSM8K, B=4'),
]:
    q = q_formula(p0, B, rho)
    n = n_avail(p0, B, rho)
    print(f"{p0:>5.2f} {B:>3d} {rho:>5.2f} {q:>8.3f} {n:>14d}     {label}")

print("\n=> ES-at-Scale's empirical N ≈ 30 floor maps to a (low-p_0, ρ ≈ 0) regime.")
print("   For our Qwen2.5-Instruct/GSM8K configurations, N_avail ≤ 4.")


## 6. The headline figures

Four single-image takeaways from the slides and the report, in the order they appear in the slide deck. Click each PNG to open it full-size; the source-of-truth versions are in [`final-slides.pdf`](final-slides.pdf) and [`final-report.pdf`](final-report.pdf).

### 6.1 CE reward is `N`-agnostic (slide: *CE reward works for all N…*)

OPT-13B fine-tuned on SST-2 with cross-entropy reward, for `N ∈ {1, 2, 4, 8, 16}` under appropriate learning-rate scaling. The curves are nearly indistinguishable — exactly the **CE population indifference** statement of Proposition 1.

![CE indifference on OPT-13B / SST-2](../plots/pop-scale-ce-opt-13b-sst2.png)

### 6.2 Binary reward is *not* `N`-agnostic (slide: *Binary reward is more realistic but does NOT work for all N*)

Qwen2.5-1.5B / GSM8K under binary accuracy reward (default normalized ES). `N = 2` collapses; `N = 4` degrades. This is the data point that motivated the workshop paper.

![Binary pop scaling, Qwen2.5-1.5B / GSM8K](../plots/pop-scale-binary-qwen-1.5b-gsm8k.png)

### 6.3 Turning normalization OFF recovers `N = 2` (Figure 1 of the report)

Same setting as 6.2 with everything held fixed except the normalization step. Normalize off (raw advantages) climbs to ≈ 0.55 val_acc; normalize on (the default) collapses to ≈ 0.02. **Lesson 3: the small-N failure is normalization, not population.**

![Normalize on vs off, Qwen2.5-1.5B / GSM8K, N=2](../results/gsm8k_norm_comparison.png)

### 6.4 ES-Malignant regime — vanilla failure mode predicts ES viability (slide: *ES-Malignant Regime: Vanilla is Dumb*)

Qwen2.5-1.5B on WIC (word-in-context). `ρ ≈ 0.88`: the base model is *frozen* — predicts "no" on almost everything — so antithetic perturbations rarely disagree, the ES gradient drifts to a self-reinforcing degenerate solution, and validation accuracy collapses below random. This is the slides' Lesson 2 — *preliminary*; not in the workshop paper.

![Malignant collapse on Qwen2.5-1.5B / WIC](../results/wic_qwen.png)

---

## 7. Optional: run the zero-training degeneracy probe on a real LLM

This is the **Appendix E** probe of the report: estimate `q̂ = P(A = 0)`, `p̂_0`, and `ρ̂` empirically from `2KB` forward passes, with **no training**. It is the canonical pre-flight check before committing to a population size.

Below we run a *tiny* version (Qwen2.5-0.5B-Instruct on SST-2, `K = 20`, `B = 4`, max 4 new tokens). On a Colab T4 this should finish in ~30 seconds. **If you don't have CUDA, the cell prints a skip message.**

In [ ]:
import os, sys, torch
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

if not torch.cuda.is_available():
    print("No CUDA available — skipping the live probe.")
    print("To run anyway on CPU (very slow):")
    print("  uv run python -m src.scripts.adhoc.probe_degeneracy \\")
    print("      --task sst2 --model Qwen/Qwen2.5-0.5B-Instruct \\")
    print("      --K 20 --batch-size 4 --max-new-tokens 4 --device cpu")
else:
    from argparse import Namespace
    from src.scripts.adhoc.probe_degeneracy import run_probe

    args = Namespace(
        task='sst2',
        model='Qwen/Qwen2.5-0.5B-Instruct',
        sigma=1e-3,
        K=20,
        batch_size=4,
        probe_size=500,
        max_new_tokens=4,
        device='cuda',
        dtype='auto',
        seed=42,
        num_workers=1,
        output=None,
        prompt_style='simple',
        no_chat_template=False,
    )
    results = run_probe(args)

    print('\n--- probe summary ---')
    print(f"p_0       (base accuracy)       = {results['p0_empirical']:.3f}")
    print(f"q̂         empirical P(A = 0)    = {results['p_degenerate_empirical']:.3f}")
    print(f"q  theory (ρ = 1/2)             = {results['p_degenerate_theory_2pi']:.3f}")
    print(f"q  theory (ρ̂)                  = {results['p_degenerate_theory_rho']:.3f}")
    print(f"ρ̂ (per-example)                = {results['rho_per_example']:.3f}")
    print(f"N_min table (δ → N_avail):        {results['nmin_table']}")


## 8. CLI cheat sheet — full-scale reproductions

Everything below requires a real GPU and takes anywhere from minutes (a single short ES run) to hours (the full `run_norm_n2_batch.sh` sweep). See [`../README.md` §8](../README.md) for the complete reference.

**Degeneracy probe (Appendix E, Table 1)**
```bash
uv run python -m src.scripts.adhoc.probe_degeneracy \
    --task gsm8k --model Qwen/Qwen2.5-1.5B-Instruct \
    --sigma 1e-3 --K 200 --batch-size 16
```

**ES fine-tuning — binary reward (Lesson 1 / Figure 2b)**
```bash
uv run python -m src.scripts.train_es \
    --task gsm8k --model Qwen/Qwen2.5-1.5B-Instruct \
    --population-size 8 --num-iters 240 --batch-size 16 \
    --sigma 1e-3 --lr 1e-3 --reward accuracy --seed 42
```

**ES fine-tuning — CE reward (Proposition 1 illustration)**
```bash
uv run python -m src.scripts.train_es \
    --task sst2 --model facebook/opt-1.3b \
    --reward ce --population-size 1 --num-iters 600 \
    --sigma 1e-3 --lr 1e-4
```

**N = 2, normalize on vs off (Figure 1)**
```bash
uv run python -m src.scripts.adhoc.run_norm_n2 \
    --model qwen2.5-1.5b-instruct --task gsm8k \
    --seeds 42 43 44 --lr-on 1e-3 --lr-off 1e-3 --sigma 1e-3
```

**Multi-model × multi-task ρ sweep (Lesson 2 table)**
```bash
uv run python -m src.scripts.adhoc.run_rho_sweep \
    --models llama3.2-1b qwen2.5-1.5b-instruct \
    --tasks mnli gsm8k wic --K 200
```

**Plotting helpers**
```bash
uv run python -m src.scripts.adhoc.plot_pop_scaling_3seeds         # Figure 2b
uv run python -m src.scripts.adhoc.plot_gsm8k_norm_comparison      # Figure 1a
uv run python -m src.scripts.adhoc.plot_norm_n2_sweep --exp-dir <sweep_dir>
```

---

## 9. Where to go next

- **The formal argument:** [`final-report.pdf`](final-report.pdf) §1–5, appendices A–H. Workshop submission.
- **The broader experimental story:** [`final-slides.pdf`](final-slides.pdf). Contains the benign/malignant taxonomy that the report leaves out.
- **Earlier development trail:** [`../.report/`](../.report/). Bi-weekly reports from Week 4 (Wordle + Prime), Week 6 (ARS reproduction), Week 8 (calibration), Week 10 (BoolQ ablations), Week 12 (the precursor SNR + N-cancellation theory).
- **Code:** the working scripts are in `../src/scripts/` and `../src/scripts/adhoc/`. The full reproduction guide and project narrative are in [`../README.md`](../README.md).

**One-sentence summary of the project:** *What looks like a tug-of-war over population size in ES is really a tug-of-war over reward magnitude — and the smallest valid `N` is set by the reward function, not by the model.*